In [41]:
import openai, json, os, requests

client = openai.OpenAI()

messages = []

In [42]:
def get_popular_movies():
    link = os.getenv("MOVIE_API_URL") + "/movies"
    response = requests.get(link)

    return response.json()

def get_movie_details(id):
    link = f"{os.getenv('MOVIE_API_URL')}/movies/{id}/credits"
    response = requests.get(link)
    return response.json()

def get_similar_movies(id):
    link = f"{os.getenv('MOVIE_API_URL')}/movies/{id}/similar"
    response = requests.get(link)
    return response.json()

FUNCTION_MAP ={
    'get_popular_movies': get_popular_movies,
    'get_movie_details': get_movie_details,
    'get_similar_movies': get_similar_movies
}

In [43]:
TOOLS = [
    {
        "type" : "function",
        "function" : {
            "name" : "get_popular_movies",
            "description" : "A function to get List popular movies",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    },
    {
        "type" : "function",
        "function" : {
            "name" : "get_movie_details",
            "description" : "Get movie details by movie id",
            "parameters" : {
                "type" : "object",
                "properties" : {
                    "id" : {
                        "type" : "integer",
                        "description" : "The ID of the movie to get details for."
                    }
                },
                "required" : ["id"]
            }   
        }
    },
    {
        "type" : "function",
        "function" : {
            "name" : "get_similar_movies",
            "description" : "Get similar movies for a movie by movie id",
            "parameters" : {
                "type" : "object",
                "properties" : {
                    "id" : {
                        "type" : "integer",
                        "description" : "The ID of the movie to get similar movies for."
                    }
                },
                "required" : ["id"]
            }   
        }
    }
]

In [44]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message :ChatCompletionMessage):
    if message.tool_calls:
        messages.append({"role": "assistant", 
                         "content": message.content or "",
                         "tool_calls": [{
                             "id": tool_call.id,
                             "type" : "function",
                                "function" : {
                                    "name" : tool_call.function.name,
                                    "arguments" : tool_call.function.arguments
                                }
                         } for tool_call in message.tool_calls]})
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}
            
            function_to_run = FUNCTION_MAP.get(function_name)
            result = function_to_run(**arguments)
            
            print(f"Agent: [{function_name}({arguments.get('id', '')}) 호출]")

            messages.append({
                "role":"tool",
                "tool_call_id":tool_call.id,
                "name"  : function_name,
                "content": json.dumps(result)
            })
            
        call_ai()
    else :
        messages.append({"role": "assistant", "content": message.content})
        print(f"Agent: {message.content}")

def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS
    ) 
    process_ai_response(response.choices[0].message)


In [45]:
while True:
    message = input("Send a message to the LLM ...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({
            "role": "user",
            "content": message
        })
        print(f"User: {message}")
        call_ai()
    
    

User: 지금 인기 있는 영화 알려줘
Agent: [get_popular_movies() 호출]
Agent: 지금 인기 있는 영화 목록은 다음과 같습니다:

1. **Peddi**
   - **개요**: 1980년대 안드라프라데시 시골에서 한 열정적인 마을 사람이 스포츠를 통해 지역 사회를 결집시켜 강력한 라이벌과 자존심을 지키기 위한 이야기입니다.
   - **개봉일**: 2026-06-03
   - **평점**: 6.7
   - ![Peddi](https://image.tmdb.org/t/p/w780/kJAJNNBYlbqAcpTDxBNnaILSMTy.jpg)

2. **Obsession**
   - **개요**: "원 위시 울"의 신비로운 전통을 깨뜨린 후 자신의 사랑을 얻기 위해 절망하는 로맨티스트가 원하는 것을 얻지만 그 욕망이 어두운 대가를 요구한다는 것을 발견하는 이야기입니다.
   - **개봉일**: 2026-05-13
   - **평점**: 7.9
   - ![Obsession](https://image.tmdb.org/t/p/w780/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg)

3. **Hai Jawani Toh Ishq Hona Hai**
   - **개요**: Jass가 conflicting priorities로 인해 결혼을 떠나고 해외에서 새로운 로맨스를 시작하지만 충격적인 폭로로 인해 사랑과 충성, 진정한 헌신의 의미를 고민하게 되는 이야기입니다.
   - **개봉일**: 2026-06-04
   - **평점**: 5.4
   - ![Hai Jawani Toh Ishq Hona Hai](https://image.tmdb.org/t/p/w780/vmlJvz6qVzYgei2V74GvnmcuQfW.jpg)

4. **The Unknown Man**
   - **개요**: Flemish 작가인 Louis가 영감을 찾기 위해 코트 다쥐르로 고립되려 하는 이야기입니다.
   - **개봉일**: 2021-10-16
   - **평점

In [46]:
while True:
    message = input("Send a message to the LLM ...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({
            "role": "user",
            "content": message
        })
        print(f"User: {message}")
        call_ai()
    
    

User: The Unknown Man에 대해 더 자세히 알려줘
Agent: [get_movie_details(879945) 호출]
Agent: **The Unknown Man (L'homme inconnu)**에 대한 자세한 정보는 다음과 같습니다:

- **개요**: Louis, a Flemish writer, decides to isolate himself at the Côte d’Azur, hoping this place will bring him inspiration.

- **개봉일**: 2021-10-16
- **평점**: 8.1
- **장르**: 드라마
- **언어**: 네덜란드어

- **출연진**:
  - **Geert Van Rampelberg** as Louis
    ![Geert Van Rampelberg](https://image.tmdb.org/t/p/w185/jECzjJvrJdBrh6xf3aRpTOu3Qxi.jpg)
  - **Samuel Suchod** as Tommy
    ![Samuel Suchod](https://image.tmdb.org/t/p/w185/m3S4EdQHuKsyRln3zeUJwuJv1yz.jpg)
  - **Anna Sacuto** as Melanie
  - **Cathy Ruiz** as Post Office Clerk

- **영화 포스터**:
![The Unknown Man](https://image.tmdb.org/t/p/w780/4TpBhdaSl5ALHbgeaYOLF8Q3haz.jpg)

이 영화는 주인공이 고립되어 새로운 영감을 찾으면서 벌어지는 이야기를 중심으로 전개됩니다. 추가적인 정보가 필요하시면 말씀해 주세요!


In [47]:
while True:
    message = input("Send a message to the LLM ...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({
            "role": "user",
            "content": message
        })
        print(f"User: {message}")
        call_ai()
    
    

User: 비슷한 영화 추천해 줄래?
Agent: [get_similar_movies(879945) 호출]
Agent: **The Unknown Man**와 비슷한 영화 몇 편을 추천해 드립니다:

1. **The Mule**
   - **개요**: 콜롬비아 여성이 프랑스 공항에 도착했을 때 수상한 행동을 보이며, 그녀의 잠금 케이스에 뭔가를 숨기고 있는 것처럼 보인다. 그러나 그녀의 가방과 몸에 대해 조사한 결과 세관에서 예상치 못한 것들이 발견된다.
   - **개봉일**: 2000-04-02
   - **평점**: 6.0
   - ![The Mule](https://image.tmdb.org/t/p/w780/A4cPtd5JEC2gujHVYi17P1IF33Y.jpg)

2. **Shutterspeed**
   - **개요**: 15세 소년 Cas는 학교의 수업을 맡은 아마추어 사진가 Jard를 위해 포즈를 취한다. 그가 둘 사이의 권력 불균형을 느끼기 시작하면서, 역전의 기회를 찾으려 한다.
   - **개봉일**: 2025-06-26
   - **평점**: 10.0
   - ![Shutterspeed](https://image.tmdb.org/t/p/w780/dmCPux38aIcOiVoQF8k2kC7NLqk.jpg)

3. **A Family Affair**
   - **개요**: 한 여성이 30번째 생일에 예상치 못한 낯선 사람과 강력한 연결을 형성하는 이야기입니다.
   - **개봉일**: 2019-06-20
   - **평점**: 6.0
   - ![A Family Affair](https://image.tmdb.org/t/p/w780/4o7oOzAzFi7wMf40HdqM6YtSzE4.jpg)

4. **Son, I Love Your Mouth 2: The Taste of the Forbidden**
   - **개요**: 아버지와 아들 간의 연애를 다룬 두 번째 이야기로, Alex가 그의 새로운 여자친구를 Gonzalo에게 소개하지만, Gonzal